## Notebook 8 — Park-Proximity Side Analysis

A side analysis, not part of the main 01→07 pipeline: it takes the exact
same cached results notebook 07 uses (nationwide OpenET, Oudin WBM,
Penman-Monteith WBM, PET-multiplier-calibrated Oudin, and flux tower data —
see notebooks 02/03/04/06) and restricts every one of them to the ~35 sites
within 50 km of a National Park (`Data/geo_data/flux_towers_near_parks.csv`,
produced by notebook 01).

**Note:** this notebook still uses notebook 04's original per-site-calibrated
Oudin run over the full 2016-2023 record, not the held-out-tested,
per-site-vs-per-ecosystem comparison notebook 07 now does nationwide (see
notebook 04's banner) — it hasn't been redesigned to match yet. Treat this
notebook's calibrated-Oudin numbers as a preliminary, in-sample look at the
park-proximity subset specifically, not the project's final nationwide
method recommendation.

**No new GEE calls, no re-running the WBM, no re-calibration** — this only
reads the same cached CSVs notebook 07 reads and filters rows by `site`.

Because this subset is small enough to facet (~35 sites, vs ~80+ nationwide),
this notebook keeps the original **full per-site faceted figures** (one panel
per site) that notebook 07 used before the nationwide expansion, rather than
the condensed one-point-per-site scatter / best-median-worst timeseries
notebook 07 now uses. Figures are saved to a separate directory
(`../Data/pet_comparison_near_parks/`) so they don't overwrite notebook 07's nationwide outputs.

For each of the three WBM AET variants (Oudin default, Oudin calibrated,
Penman-Monteith), this produces:
- Per-site bias (MBE), MAE, RMSE, R², and slope vs. OpenET and vs. flux tower
- Faceted 1:1 scatter plots (WBM AET vs. OpenET, one panel per park-adjacent site)
- Faceted monthly AET timeseries (one panel per park-adjacent site)
- A grouped bar chart of mean RMSE by ecosystem
- An interactive spatial map of which variant best matches OpenET at each site

**Prerequisites:** run notebooks 01→02→03→04→06 at least once each (so the
nationwide caches this notebook filters actually exist).

# Setup Workspace

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines

pd.set_option("display.width", 120)
print("Imports OK -- this notebook only reads cached CSVs (filtered to park-proximity sites), no GEE/raster access needed.")


### Load cached prerequisites, filtered to the park-proximity subset

Reads the same nationwide caches notebook 07 reads (notebooks 02, 03, 04, 06), then restricts every one of them to the sites in `Data/geo_data/flux_towers_near_parks.csv`.

In [ ]:
# ── Load cached outputs from notebooks 02 (OpenET), 03 (Oudin WBM run), 04
# (calibrated Oudin), and 06 (Penman-Monteith WBM run) -- the same nationwide
# caches notebook 07 reads -- then restrict every frame to the sites within
# 50 km of a National Park. No GEE/raster access, no re-running the WBM. ─────
flux_towers_all        = pd.read_csv("../Data/geo_data/flux_towers.csv")
flux_towers_near_parks = pd.read_csv("../Data/geo_data/flux_towers_near_parks.csv")
park_sites = set(flux_towers_near_parks["site"])

flux_towers = (
    flux_towers_all[flux_towers_all["site"].isin(park_sites)]
    .reset_index(drop=True)
)
print(f"Restricting nationwide caches to {len(flux_towers)} / {len(flux_towers_all)} "
      f"sites within 50 km of a National Park.")

openet_gridcell_df = pd.read_csv(
    "../Data/open_et_gridcell/openet_gridcell_timeseries.csv",
    parse_dates=["date"]
)
openet_gridcell_df = (
    openet_gridcell_df[openet_gridcell_df["site"].isin(park_sites)]
    .reset_index(drop=True)
)

wbm_results_oudin = pd.read_csv(
    "../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
    parse_dates=["date"]
)
wbm_results_oudin = (
    wbm_results_oudin[wbm_results_oudin["site"].isin(park_sites)]
    .reset_index(drop=True)
)

wbm_results_pm = pd.read_csv(
    "../Data/gridmet_cache/wbm_results_penman_monteith.csv",
    parse_dates=["date"]
)
wbm_results_pm = (
    wbm_results_pm[wbm_results_pm["site"].isin(park_sites)]
    .reset_index(drop=True)
)

# Both WBM caches were produced with the repo's default TO_INCHES = True (see
# the "global model settings" cell in notebooks 03/06) -- set here to match,
# so AET can be converted to mm below for comparison against OpenET.
TO_INCHES = True

# Defensive fallback if either cache is missing ecosystem/state (e.g. an older
# cache from before those columns were tagged during the WBM run loop).
for _name in ("wbm_results_oudin", "wbm_results_pm"):
    _df = globals()[_name]
    if "ecosystem" not in _df.columns:
        globals()[_name] = _df.merge(
            flux_towers[["site", "ecosystem", "state"]].drop_duplicates(),
            on="site", how="left"
        )

# flux_monthly isn't cached (it's a cheap local read, no GEE call) -- rebuild
# it for just the park-proximity sites.
flux_frames = []
for row in flux_towers.itertuples():
    fpath = f"../Data/flux_ET_dataset/monthly_data_files/{row.site}_monthly_data.csv"
    if not os.path.exists(fpath):
        continue
    _fdf = pd.read_csv(fpath, parse_dates=["date"])
    _fdf = _fdf[_fdf["date"] >= "2016-01-01"]
    _fdf["site"] = row.site
    flux_frames.append(_fdf)
flux_monthly = (
    pd.concat(flux_frames, ignore_index=True)
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

# Notebook 04's per-site PET-multiplier-calibrated Oudin run -- cached
# already monthly (with a precomputed aet_mm column in mm regardless of
# TO_INCHES), keyed by whatever OBJECTIVE notebook 04 was last run with.
# Filtered to park sites.
#
# NOTE: this cache is now fit on notebook 04's 2016-2021 calibration period
# only (see notebook 04's CAL_START/CAL_END), not the full 2016-2023 record
# -- this notebook, unlike notebook 07, does not yet restrict its own
# metrics to the 2022-2023 held-out period, so treat its calibrated-Oudin
# numbers as a mix of in-sample and out-of-sample months. See notebook 07
# for the held-out, robustness-focused comparison across all four variants.
with open("../Data/gridmet_cache/last_objective.txt") as f:
    CAL_OBJECTIVE = f.read().strip()
wbm_monthly_oudin_cal = pd.read_csv(
    f"../Data/gridmet_cache/wbm_monthly_site_cal_{CAL_OBJECTIVE}.csv",
    parse_dates=["date_monthly"]
)
wbm_monthly_oudin_cal = (
    wbm_monthly_oudin_cal[wbm_monthly_oudin_cal["site"].isin(park_sites)]
    .reset_index(drop=True)
)

print(f"flux_towers (park-proximity) : {flux_towers.shape}")
print(f"openet_gridcell_df           : {openet_gridcell_df.shape}")
print(f"wbm_results_oudin            : {wbm_results_oudin.shape}")
print(f"wbm_results_pm               : {wbm_results_pm.shape}")
print(f"wbm_monthly_oudin_cal        : {wbm_monthly_oudin_cal.shape}  (OBJECTIVE='{CAL_OBJECTIVE}')")
print(f"flux_monthly                 : {flux_monthly.shape}")


### Step 1 — Aggregate both WBM runs to monthly AET

Same sum/mean aggregation rule notebooks 03/04/06 use (fluxes summed, states
averaged), converting AET to mm for a fair comparison against OpenET.

In [ ]:
FLUX_COLS  = ["ppt_mm", "RAIN", "SNOW", "MELT", "AET", "RUNOFF", "D"]
STATE_COLS = ["SOIL", "PACK", "tmean_C"]


def to_monthly(df, flux_cols=FLUX_COLS, state_cols=STATE_COLS):
    """Aggregate daily WBM output to monthly (sum fluxes, mean states)."""
    agg_dict = {c: "sum" for c in flux_cols if c in df.columns}
    agg_dict.update({c: "mean" for c in state_cols if c in df.columns})
    return (
        df
        .assign(date_monthly=lambda d: pd.to_datetime(
            d["date"].dt.to_period("M").dt.to_timestamp()))
        .groupby(["site", "ecosystem", "state", "date_monthly"], as_index=False)
        .agg(agg_dict)
        .sort_values(["site", "date_monthly"])
        .reset_index(drop=True)
    )


wbm_monthly_oudin = to_monthly(wbm_results_oudin)
wbm_monthly_pm    = to_monthly(wbm_results_pm)

_scale = 25.4 if TO_INCHES else 1.0
wbm_monthly_oudin["aet_oudin_mm"] = wbm_monthly_oudin["AET"] * _scale
wbm_monthly_pm["aet_pm_mm"]       = wbm_monthly_pm["AET"] * _scale

print(f"Oudin monthly : {wbm_monthly_oudin.shape}")
print(f"PM monthly    : {wbm_monthly_pm.shape}")


### Step 2 — Merge all three WBM AET variants with OpenET (4 km gridcell ensemble)

Inner-joined on site + month, so all three variants are compared over the exact
same set of site-months (a like-for-like comparison).

In [ ]:
def to_month_start(df, date_col="date"):
    """Normalise a date column to the first day of its month (in-place copy)."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.to_period("M").dt.to_timestamp()
    return df


openet_gc_plot = to_month_start(openet_gridcell_df).rename(
    columns={"et_ensemble_mm": "et_ensemble_gc_mm"}
)[["site", "date", "et_ensemble_gc_mm"]]

oudin_plot = wbm_monthly_oudin[["site", "ecosystem", "date_monthly", "aet_oudin_mm"]].rename(
    columns={"date_monthly": "date"}
)
oudin_cal_plot = wbm_monthly_oudin_cal[["site", "date_monthly", "aet_mm"]].rename(
    columns={"date_monthly": "date", "aet_mm": "aet_oudin_cal_mm"}
)
pm_plot = wbm_monthly_pm[["site", "date_monthly", "aet_pm_mm"]].rename(
    columns={"date_monthly": "date"}
)

combined = (
    openet_gc_plot
    .merge(oudin_plot,     on=["site", "date"], how="inner")
    .merge(oudin_cal_plot, on=["site", "date"], how="inner")
    .merge(pm_plot,        on=["site", "date"], how="inner")
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

print(f"Matched site-months (OpenET \u2229 Oudin \u2229 Oudin-calibrated \u2229 Penman-Monteith): {len(combined):,}")
print(combined.head(6).to_string(index=False))


### Step 3 — Per-site metrics: bias / RMSE / R² vs. OpenET

In [ ]:
def compute_metrics(df, obs_col, pred_col, site_col="site", min_n=6):
    """Per-site slope (through origin), MBE, MAE, RMSE, R² -- same definition
    used in notebook 04's calibration workflow, applied here uncalibrated."""
    def _slope(o, p): return np.dot(o, p) / np.dot(o, o)
    def _mbe(o, p):   return float(np.mean(p - o))
    def _mae(o, p):   return float(np.mean(np.abs(p - o)))
    def _rmse(o, p):  return float(np.sqrt(np.mean((p - o) ** 2)))
    def _r2(o, p):
        ss_res = np.sum((o - p) ** 2)
        ss_tot = np.sum((o - np.mean(o)) ** 2)
        return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    records = []
    for site, grp in df.groupby(site_col):
        paired = grp[[obs_col, pred_col]].dropna()
        n = len(paired)
        if n < min_n:
            records.append({"site": site, "n": n, "slope": np.nan, "MBE": np.nan,
                             "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                             "note": f"insufficient data (n={n})"})
            continue
        obs, pred = paired[obs_col].values, paired[pred_col].values
        records.append({
            "site": site, "n": n,
            "slope": round(_slope(obs, pred), 3),
            "MBE":   round(_mbe(obs, pred), 3),
            "MAE":   round(_mae(obs, pred), 3),
            "RMSE":  round(_rmse(obs, pred), 3),
            "R2":    round(_r2(obs, pred), 3),
            "note":  "",
        })
    return pd.DataFrame(records).sort_values("site").reset_index(drop=True)


metrics_oudin_vs_openet     = compute_metrics(combined, obs_col="et_ensemble_gc_mm", pred_col="aet_oudin_mm")
metrics_oudin_cal_vs_openet = compute_metrics(combined, obs_col="et_ensemble_gc_mm", pred_col="aet_oudin_cal_mm")
metrics_pm_vs_openet        = compute_metrics(combined, obs_col="et_ensemble_gc_mm", pred_col="aet_pm_mm")

for label, mdf in [("Oudin WBM vs OpenET (4 km gridcell)", metrics_oudin_vs_openet),
                    (f"Calibrated Oudin WBM vs OpenET (4 km gridcell, {CAL_OBJECTIVE})", metrics_oudin_cal_vs_openet),
                    ("Penman-Monteith WBM vs OpenET (4 km gridcell)", metrics_pm_vs_openet)]:
    valid = mdf[mdf["note"] == ""]
    print(f"\n\u2500\u2500 {label} \u2500\u2500" * 1)
    print(mdf[["site", "n", "slope", "MBE", "MAE", "RMSE", "R2", "note"]].to_string(index=False))
    if not valid.empty:
        print(f"  mean  : slope={valid['slope'].mean():.3f}  MBE={valid['MBE'].mean():.2f}  "
              f"RMSE={valid['RMSE'].mean():.2f}  R2={valid['R2'].mean():.3f}  (n={len(valid)} sites)")

os.makedirs("../Data/pet_comparison_near_parks", exist_ok=True)
metrics_oudin_vs_openet.to_csv("../Data/pet_comparison_near_parks/oudin_vs_openet_metrics.csv", index=False)
metrics_oudin_cal_vs_openet.to_csv("../Data/pet_comparison_near_parks/oudin_cal_vs_openet_metrics.csv", index=False)
metrics_pm_vs_openet.to_csv("../Data/pet_comparison_near_parks/pm_vs_openet_metrics.csv", index=False)
combined.to_csv("../Data/pet_comparison_near_parks/oudin_pm_openet_monthly_values.csv", index=False)
print("\nSaved to Data/pet_comparison_near_parks/{oudin_vs_openet_metrics,oudin_cal_vs_openet_metrics,"
      "pm_vs_openet_metrics,oudin_pm_openet_monthly_values}.csv")


### Step 4 — Figures

In [ ]:
def make_scatter_facets(combined_df, metrics_df, pred_col, pred_label, title,
                         save_path, n_cols=4):
    """Faceted 1:1 scatter: WBM AET (pred_col) vs OpenET ensemble, one panel
    per site, annotated with each site's R\u00b2 from metrics_df."""
    sites_list = sorted(combined_df["site"].unique())
    n_rows = int(np.ceil(len(sites_list) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, n_rows * 2.1),
                              constrained_layout=True)
    axes_flat = np.atleast_1d(axes).flatten()

    for ax, site in zip(axes_flat, sites_list):
        sub = combined_df[combined_df["site"] == site].dropna(
            subset=["et_ensemble_gc_mm", pred_col])
        m = metrics_df[metrics_df["site"] == site]

        ax.scatter(sub["et_ensemble_gc_mm"], sub[pred_col],
                   s=18, color="steelblue", alpha=0.7,
                   edgecolors="k", linewidths=0.3)

        if not sub.empty:
            lims = [0, max(sub["et_ensemble_gc_mm"].max(), sub[pred_col].max()) * 1.05]
            ax.plot(lims, lims, color="grey", linestyle="--", linewidth=1, zorder=1)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        r2v = m["R2"].iloc[0] if not m.empty and m["note"].iloc[0] == "" else np.nan
        ax.annotate(f"R\u00b2={r2v:.2f}" if pd.notna(r2v) else "n/a",
                    xy=(0.05, 0.9), xycoords="axes fraction", fontsize=7,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                              alpha=0.7, edgecolor="none"))
        ax.set_title(site, fontsize=9, fontweight="bold", pad=3)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)

    for ax in axes_flat[len(sites_list):]:
        ax.set_visible(False)

    fig.supxlabel("OpenET ensemble, 4 km gridcell (mm/month)", fontsize=9)
    fig.supylabel(pred_label, fontsize=9)
    fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")


make_scatter_facets(
    combined, metrics_oudin_vs_openet, pred_col="aet_oudin_mm",
    pred_label="Oudin WBM AET (mm/month)",
    title="Oudin-driven WBM AET vs OpenET (4 km gridcell)",
    save_path="../Data/pet_comparison_near_parks/scatter_oudin_vs_openet.png",
)


In [ ]:
make_scatter_facets(
    combined, metrics_oudin_cal_vs_openet, pred_col="aet_oudin_cal_mm",
    pred_label=f"Calibrated Oudin WBM AET (mm/month, {CAL_OBJECTIVE})",
    title=f"Calibrated Oudin-driven WBM AET vs OpenET (4 km gridcell, {CAL_OBJECTIVE})",
    save_path="../Data/pet_comparison_near_parks/scatter_oudin_cal_vs_openet.png",
)


In [ ]:
make_scatter_facets(
    combined, metrics_pm_vs_openet, pred_col="aet_pm_mm",
    pred_label="Penman-Monteith WBM AET (mm/month)",
    title="Penman-Monteith-driven WBM AET vs OpenET (4 km gridcell)",
    save_path="../Data/pet_comparison_near_parks/scatter_pm_vs_openet.png",
)


In [ ]:
eco_lookup = combined[["site", "ecosystem"]].drop_duplicates()

eco_rmse = (
    metrics_oudin_vs_openet[metrics_oudin_vs_openet["note"] == ""]
    [["site", "RMSE"]].rename(columns={"RMSE": "RMSE_oudin"})
    .merge(
        metrics_oudin_cal_vs_openet[metrics_oudin_cal_vs_openet["note"] == ""]
        [["site", "RMSE"]].rename(columns={"RMSE": "RMSE_oudin_cal"}),
        on="site", how="outer"
    )
    .merge(
        metrics_pm_vs_openet[metrics_pm_vs_openet["note"] == ""]
        [["site", "RMSE"]].rename(columns={"RMSE": "RMSE_pm"}),
        on="site", how="outer"
    )
    .merge(eco_lookup, on="site", how="left")
)

eco_summary = (
    eco_rmse.groupby("ecosystem")[["RMSE_oudin", "RMSE_oudin_cal", "RMSE_pm"]]
    .mean().round(2).reset_index()
)
print("Mean RMSE vs OpenET by ecosystem (mm/month):")
print(eco_summary.to_string(index=False))

ecosystems = eco_summary["ecosystem"].tolist()
x = np.arange(len(ecosystems))
w = 0.27

fig, ax = plt.subplots(figsize=(7.5, 4), constrained_layout=True)
ax.bar(x - w, eco_summary["RMSE_oudin"], width=w, color="darkblue",
       label="Oudin (default)", edgecolor="white")
ax.bar(x, eco_summary["RMSE_oudin_cal"], width=w, color="seagreen",
       label=f"Oudin (calibrated, {CAL_OBJECTIVE})", edgecolor="white")
ax.bar(x + w, eco_summary["RMSE_pm"], width=w, color="tomato",
       label="Penman-Monteith", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(ecosystems, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean RMSE vs OpenET (mm/month)", fontsize=9)
ax.set_title("WBM AET RMSE vs OpenET, by ecosystem\n"
             "(Oudin default & Penman-Monteith uncalibrated; Kc = 1.0 for Penman-Monteith)",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig("../Data/pet_comparison_near_parks/rmse_by_ecosystem_oudin_vs_pm.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved to Data/pet_comparison_near_parks/rmse_by_ecosystem_oudin_vs_pm.png")


### Step 5 — Timeseries: OpenET, Oudin (default + calibrated), Penman-Monteith, and Flux tower, per site

Faceted monthly AET timeseries (one panel per site) showing OpenET (4 km
gridcell ensemble), all three WBM runs, and the flux tower's own AET together.

In [ ]:
# ── Timeseries: OpenET, Oudin AET, Penman-Monteith AET, and Flux tower AET, per site ──
flux_plot = to_month_start(
    flux_monthly[["site", "date", "ET_corr"]].rename(columns={"ET_corr": "aet_flux_mm"})
)

combined_ts = (
    openet_gc_plot
    .merge(oudin_plot,     on=["site", "date"], how="outer")
    .merge(oudin_cal_plot, on=["site", "date"], how="outer")
    .merge(pm_plot,        on=["site", "date"], how="outer")
    .merge(flux_plot,      on=["site", "date"], how="outer")
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

sites_list = sorted(combined_ts["site"].unique())
n_cols = 4
n_rows = int(np.ceil(len(sites_list) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, n_rows * 1.8),
                          sharey=False, constrained_layout=True)
axes_flat = np.atleast_1d(axes).flatten()

for ax, site in zip(axes_flat, sites_list):
    sub = combined_ts[combined_ts["site"] == site].sort_values("date")

    ax.plot(sub["date"], sub["et_ensemble_gc_mm"],
            color="#3f3f3f", linewidth=1.3, linestyle="-",
            label="OpenET ensemble (4 km gridcell)", zorder=3)
    ax.plot(sub["date"], sub["aet_oudin_mm"],
            color="darkblue", linewidth=1.3, linestyle="-",
            label="WBM AET (Oudin, default)", zorder=4)
    ax.plot(sub["date"], sub["aet_oudin_cal_mm"],
            color="seagreen", linewidth=1.3, linestyle="-",
            label=f"WBM AET (Oudin, calibrated, {CAL_OBJECTIVE})", zorder=4.5)
    ax.plot(sub["date"], sub["aet_pm_mm"],
            color="tomato", linewidth=1.3, linestyle="-",
            label="WBM AET (Penman-Monteith)", zorder=5)

    flux_sub = sub.dropna(subset=["aet_flux_mm"])
    if not flux_sub.empty:
        ax.scatter(flux_sub["date"], flux_sub["aet_flux_mm"],
                   color="black", s=14, zorder=8, label="Flux tower ET")

    ax.set_title(site, fontsize=9, fontweight="bold", pad=3)
    ax.set_ylabel("AET (mm/month)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

for ax in axes_flat[len(sites_list):]:
    ax.set_visible(False)

if len(sites_list) < len(axes_flat):
    legend_ax = axes_flat[len(sites_list)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")
    legend_ax.legend(
        handles=[
            mlines.Line2D([], [], color="#3f3f3f", linewidth=1.3,
                          label="OpenET ensemble (4 km gridcell)"),
            mlines.Line2D([], [], color="darkblue", linewidth=1.3,
                          label="WBM AET (Oudin, default)"),
            mlines.Line2D([], [], color="seagreen", linewidth=1.3,
                          label=f"WBM AET (Oudin, calibrated, {CAL_OBJECTIVE})"),
            mlines.Line2D([], [], color="tomato", linewidth=1.3,
                          label="WBM AET (Penman-Monteith)"),
            mlines.Line2D([], [], color="black", linewidth=0, marker="o",
                          markersize=5, label="Flux tower ET"),
        ],
        loc="center", fontsize=8, frameon=False,
        title="Data sources", title_fontsize=9,
    )

fig.suptitle("Monthly AET timeseries \u2014 OpenET vs Oudin (default + calibrated) vs Penman-Monteith vs Flux tower",
             fontsize=12, fontweight="bold")
plt.savefig("../Data/pet_comparison_near_parks/timeseries_openet_oudin_pm_flux.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved to Data/pet_comparison_near_parks/timeseries_openet_oudin_pm_flux.png")


### Step 6 — Per-site metrics: bias / RMSE / R² vs. Flux tower

Same metric definitions as Step 3, applied to `combined_ts` (built in Step 5,
so this also picks up any site-months present for flux tower / WBM but
missing from the OpenET merge).

In [ ]:
metrics_oudin_vs_flux     = compute_metrics(combined_ts, obs_col="aet_flux_mm", pred_col="aet_oudin_mm")
metrics_oudin_cal_vs_flux = compute_metrics(combined_ts, obs_col="aet_flux_mm", pred_col="aet_oudin_cal_mm")
metrics_pm_vs_flux        = compute_metrics(combined_ts, obs_col="aet_flux_mm", pred_col="aet_pm_mm")

for label, mdf in [("Oudin WBM vs Flux tower", metrics_oudin_vs_flux),
                    (f"Calibrated Oudin WBM vs Flux tower ({CAL_OBJECTIVE})", metrics_oudin_cal_vs_flux),
                    ("Penman-Monteith WBM vs Flux tower", metrics_pm_vs_flux)]:
    valid = mdf[mdf["note"] == ""]
    print(f"\n\u2500\u2500 {label} \u2500\u2500")
    print(mdf[["site", "n", "slope", "MBE", "MAE", "RMSE", "R2", "note"]].to_string(index=False))
    if not valid.empty:
        print(f"  mean  : slope={valid['slope'].mean():.3f}  MBE={valid['MBE'].mean():.2f}  "
              f"RMSE={valid['RMSE'].mean():.2f}  R2={valid['R2'].mean():.3f}  (n={len(valid)} sites)")

metrics_oudin_vs_flux.to_csv("../Data/pet_comparison_near_parks/oudin_vs_flux_metrics.csv", index=False)
metrics_oudin_cal_vs_flux.to_csv("../Data/pet_comparison_near_parks/oudin_cal_vs_flux_metrics.csv", index=False)
metrics_pm_vs_flux.to_csv("../Data/pet_comparison_near_parks/pm_vs_flux_metrics.csv", index=False)
print("\nSaved to Data/pet_comparison_near_parks/{oudin_vs_flux_metrics,oudin_cal_vs_flux_metrics,"
      "pm_vs_flux_metrics}.csv")


### Step 7 — Spatial map: which WBM AET variant matches OpenET best, by site?

For each site, finds whichever of the three variants (Oudin default, Oudin
calibrated, Penman-Monteith) has the lowest RMSE vs. OpenET (from Step 3),
and maps that as a category, sized by how much better it is than the
runner-up at that site.

This is an **interactive** Plotly map rather than a static one: it uses
Plotly's built-in USA state outlines (no shapefile download, so no
dependency on Census/GitHub being reachable), and hovering a point shows
its full metrics instead of a permanent text label -- which avoids the
overlapping-label problem the static version had at clustered sites (e.g.
the LYS_* and US-Ro* groups).

In [ ]:
import plotly.graph_objects as go

# ── Merge each variant's RMSE (vs OpenET) with site coordinates ───────────────
map_df = (
    metrics_oudin_vs_openet[metrics_oudin_vs_openet["note"] == ""][["site", "RMSE"]]
    .rename(columns={"RMSE": "RMSE_oudin"})
    .merge(
        metrics_oudin_cal_vs_openet[metrics_oudin_cal_vs_openet["note"] == ""][["site", "RMSE"]]
        .rename(columns={"RMSE": "RMSE_oudin_cal"}),
        on="site", how="inner"
    )
    .merge(
        metrics_pm_vs_openet[metrics_pm_vs_openet["note"] == ""][["site", "RMSE"]]
        .rename(columns={"RMSE": "RMSE_pm"}),
        on="site", how="inner"
    )
    .merge(flux_towers[["site", "x", "y", "ecosystem"]], on="site", how="left")
    .dropna(subset=["x", "y"])
    .reset_index(drop=True)
)

RMSE_COLS = ["RMSE_oudin", "RMSE_oudin_cal", "RMSE_pm"]
METHOD_LABELS = {
    "RMSE_oudin":     "Oudin (default)",
    "RMSE_oudin_cal": f"Oudin (calibrated, {CAL_OBJECTIVE})",
    "RMSE_pm":        "Penman-Monteith",
}
METHOD_COLORS = {
    "Oudin (default)":                        "darkblue",
    f"Oudin (calibrated, {CAL_OBJECTIVE})":    "seagreen",
    "Penman-Monteith":                         "tomato",
}

best_col = map_df[RMSE_COLS].idxmin(axis=1)
map_df["best_method"] = best_col.map(METHOD_LABELS)
map_df["best_RMSE"]   = map_df[RMSE_COLS].min(axis=1)
sorted_vals = map_df[RMSE_COLS].apply(lambda r: sorted(r.values), axis=1)
map_df["margin"] = sorted_vals.apply(lambda v: v[1] - v[0])

print(f"Sites mapped: {len(map_df)}")
print(map_df[["site", "ecosystem"] + RMSE_COLS + ["best_method", "margin"]]
      .round(2).to_string(index=False))
print()
print("Sites where each variant is best:")
print(map_df["best_method"].value_counts().to_string())

# ── Interactive Plotly map -- scope="usa" draws state outlines from Plotly's
# own bundled map data, so this needs no shapefile download at all ───────────
max_margin = max(map_df["margin"].max(), 1)

fig = go.Figure()
for method, color in METHOD_COLORS.items():
    sub = map_df[map_df["best_method"] == method]
    if sub.empty:
        continue
    sizes = (sub["margin"] / max_margin * 26).clip(lower=6)
    hover_text = [
        f"<b>{row.site}</b> ({row.ecosystem})<br>"
        f"Oudin (default): {row.RMSE_oudin:.1f} mm/mo<br>"
        f"Oudin (calibrated): {row.RMSE_oudin_cal:.1f} mm/mo<br>"
        f"Penman-Monteith: {row.RMSE_pm:.1f} mm/mo<br>"
        f"Best: {row.best_method} (margin {row.margin:.1f} mm/mo)"
        for row in sub.itertuples()
    ]
    fig.add_trace(go.Scattergeo(
        lon=sub["x"], lat=sub["y"],
        mode="markers",
        name=method,
        marker=dict(size=sizes, color=color, line=dict(width=1, color="black"), opacity=0.85),
        text=hover_text,
        hoverinfo="text",
    ))

fig.update_geos(
    scope="usa",
    showland=True, landcolor="whitesmoke",
    showlakes=True, lakecolor="white",
    subunitcolor="grey", subunitwidth=0.6,
    countrycolor="grey",
)
fig.update_layout(
    title="Which WBM AET variant matches OpenET best, by site?"
          "<br><sup>marker size \u221d margin over the runner-up -- hover for per-site RMSE</sup>",
    legend_title_text="Best-matching variant",
    margin=dict(l=0, r=0, t=60, b=0),
    height=550,
)

os.makedirs("../Data/pet_comparison_near_parks", exist_ok=True)
fig.write_html("../Data/pet_comparison_near_parks/map_best_method_vs_openet.html")
print("Saved interactive map to Data/pet_comparison_near_parks/map_best_method_vs_openet.html")

# Best-effort static PNG too (needs kaleido + a local Chrome install; skipped
# gracefully if unavailable so this cell never fails because of it).
try:
    fig.write_image("../Data/pet_comparison_near_parks/map_best_method_vs_openet.png",
                     width=1000, height=650, scale=2)
    print("Saved static PNG to Data/pet_comparison_near_parks/map_best_method_vs_openet.png")
except Exception as e:
    print(f"(skipped static PNG export -- {e})")

fig.show()


### Summary

This is the park-proximity counterpart to notebook 07's nationwide benchmark:
same three WBM AET variants (default Oudin, calibrated Oudin, Penman-Monteith)
against OpenET and flux tower ET, restricted to the ~35 sites within 50 km of
a National Park, with full per-site facets since the subset is small enough
to read at that scale. All the same caveats and follow-ups from notebook 07's
Summary apply here too (Penman-Monteith isn't calibrated yet, `Kc = 1.0` is a
baseline assumption, etc.) -- see notebook 07 for details, since this notebook
doesn't duplicate that discussion.

If a different calibration objective is preferred, re-run notebook 04 with a
different `OBJECTIVE` before re-running this notebook -- the prereq-loading
cell above picks up whatever `OBJECTIVE` notebook 04 was last cached with.